In [ ]:
from partial_discharge_adaptive_fusion.protocol import assert_protocol_frozen, load_experiment_config
CONFIG = load_experiment_config('configs/experiments/two-dataset-confirmatory-v2-batch4-localraw.yaml')
assert_protocol_frozen(CONFIG)
assert CONFIG['experts']['batch_size'] == CONFIG['experts']['inference_batch_size'] == 4
print('Using config:', CONFIG['config_version'])

# Paired statistical analysis

**Objective.** Quantify fixed, adaptive and conservative effects per dataset and seed.

**Inputs.** Locked v2-batch4 predictions and labels; no model-selection inputs.

**Outputs.** Paired bootstrap MCC deltas, 95% intervals, McNemar counts, seed summaries and error transitions.

**Experimental role.** Inferential analysis.

**Leakage constraints.** Bootstrap samples are paired within seed; repeated seeds on the same test rows are not treated as independent test observations; all input files must share the v2 `config_version`.

In [ ]:
from partial_discharge_adaptive_fusion.evaluation import mcnemar_counts, paired_bootstrap_delta, summarize_seed_deltas

print('Required comparisons: Fixed-Temporal, Adaptive-Fixed, Conservative-Fixed, Adaptive-Temporal')
print('Bootstrap iterations per seed:', 10000)
print('Primary metric: MCC')

def paired_comparison(labels, prediction_a, prediction_b, seed):
    bootstrap = paired_bootstrap_delta(labels, prediction_a, prediction_b, iterations=10000, seed=seed)
    mcnemar = mcnemar_counts(labels, prediction_a, prediction_b)
    return {**bootstrap, **mcnemar, 'seed': seed}

def summarize_comparison(seed_rows):
    return summarize_seed_deltas(row['point_estimate'] for row in seed_rows)


## Findings and handoff

Report point estimates, intervals and seed-level consistency together. Statistical significance alone does not replace the pre-specified effect criterion.

**Next stage:** synthesize the two datasets without pooling their raw probabilities.